## Code Flow

1. Load the Dataset

2. Basic Preprocessing

3. Training Process
    - Create the Model
    - Forward Pass
    - Loss Calculation
    - Backpropagation
    - Parameters Update

4. Model Evaluation

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [3]:
df.shape

(569, 33)

In [4]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)

In [6]:
df.head()

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['diagnosis']), df['diagnosis'], test_size=0.2, random_state=42)

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [9]:
print(X_train.shape)
print(X_test.shape)

(455, 30)
(114, 30)


In [10]:
print(y_train.shape)
print(y_test.shape)

(455,)
(114,)


In [11]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [12]:
y_train[:5], y_test[:5]

(array([0, 1, 0, 0, 0]), array([0, 1, 1, 0, 0]))

In [13]:
# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_train)
X_test_tensor = torch.from_numpy(X_test)
y_train_tensor = torch.from_numpy(y_train)
y_test_tensor = torch.from_numpy(y_test)

In [14]:
# Convert tensors to float32
X_train_tensor = X_train_tensor.float()
X_test_tensor = X_test_tensor.float()
y_train_tensor = y_train_tensor.float()
y_test_tensor = y_test_tensor.float()

In [15]:
X_train_tensor[1].dtype

torch.float32

In [16]:
print(X_train_tensor.shape)
print(X_test_tensor.shape)
print(y_train_tensor.shape)
print(y_test_tensor.shape)

torch.Size([455, 30])
torch.Size([114, 30])
torch.Size([455])
torch.Size([114])


In [18]:
# Create Custom Dataset
from torch.utils.data import Dataset, DataLoader

class BreastCancerDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [19]:
train_dataset = BreastCancerDataset(X_train_tensor, y_train_tensor)
test_dataset = BreastCancerDataset(X_test_tensor, y_test_tensor)

In [20]:
train_dataset[0]

(tensor([-1.4408, -0.4353, -1.3621, -1.1391,  0.7806,  0.7189,  2.8231, -0.1191,
          1.0927,  2.4582, -0.2638, -0.0161, -0.4704, -0.4748,  0.8384,  3.2510,
          8.4389,  3.3920,  2.6212,  2.0612, -1.2329, -0.4763, -1.2479, -0.9740,
          0.7229,  1.1867,  4.6728,  0.9320,  2.0972,  1.8865]),
 tensor(0.))

In [21]:
# Create DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [22]:
# Define the model
class SimpleNN(nn.Module):
    def __init__(self, X):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)

In [23]:
learning_rate = 0.01
epochs = 100

## Training Pipeline

In [24]:
# Initialize the model and optimizer
model = SimpleNN(X_train_tensor)

# Define the optimizer
# Using SGD optimizer
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

# Define the loss function
loss_fn = nn.BCELoss()

In [25]:
# Training loop
for epoch in range(epochs):
    model.train()
    for batch_features, batch_labels in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_features)
        loss = loss_fn(outputs.squeeze(), batch_labels)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

Epoch 1/100, Loss: 0.7070
Epoch 2/100, Loss: 0.6944
Epoch 3/100, Loss: 0.6595
Epoch 4/100, Loss: 0.6073
Epoch 5/100, Loss: 0.6889
Epoch 6/100, Loss: 0.5850
Epoch 7/100, Loss: 0.5839
Epoch 8/100, Loss: 0.5769
Epoch 9/100, Loss: 0.6076
Epoch 10/100, Loss: 0.5907
Epoch 11/100, Loss: 0.5531
Epoch 12/100, Loss: 0.5396
Epoch 13/100, Loss: 0.3662
Epoch 14/100, Loss: 0.4778
Epoch 15/100, Loss: 0.5099
Epoch 16/100, Loss: 0.4622
Epoch 17/100, Loss: 0.3796
Epoch 18/100, Loss: 0.2441
Epoch 19/100, Loss: 0.4820
Epoch 20/100, Loss: 0.2467
Epoch 21/100, Loss: 0.2640
Epoch 22/100, Loss: 0.1393
Epoch 23/100, Loss: 0.1668
Epoch 24/100, Loss: 0.1233
Epoch 25/100, Loss: 0.4093
Epoch 26/100, Loss: 0.1063
Epoch 27/100, Loss: 0.2800
Epoch 28/100, Loss: 0.2318
Epoch 29/100, Loss: 0.0922
Epoch 30/100, Loss: 0.3198
Epoch 31/100, Loss: 0.2352
Epoch 32/100, Loss: 0.4025
Epoch 33/100, Loss: 0.0614
Epoch 34/100, Loss: 0.0430
Epoch 35/100, Loss: 0.2407
Epoch 36/100, Loss: 0.1169
Epoch 37/100, Loss: 0.0991
Epoch 38/1

In [26]:
# Testing loop
model.eval()
correct = 0
total = 0
with torch.inference_mode():
    for batch_features, batch_labels in test_loader:
        outputs = model(batch_features)
        predicted = (outputs.squeeze() > 0.5).float()
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9825


In [27]:
from torchinfo import summary
summary(model, input_size = (X_train_tensor.shape[0], X_train_tensor.shape[1]), col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"], row_settings=["var_names"], depth=2, device="cpu")

Layer (type (var_name))                  Input Shape               Output Shape              Param #                   Kernel Shape              Mult-Adds
SimpleNN (SimpleNN)                      [455, 30]                 [455, 1]                  --                        --                        --
├─Sequential (model)                     [455, 30]                 [455, 1]                  --                        --                        --
│    └─Linear (0)                        [455, 30]                 [455, 64]                 1,984                     --                        902,720
│    └─ReLU (1)                          [455, 64]                 [455, 64]                 --                        --                        --
│    └─Linear (2)                        [455, 64]                 [455, 32]                 2,080                     --                        946,400
│    └─ReLU (3)                          [455, 32]                 [455, 32]                 --

In [26]:
model.parameters

<bound method Module.parameters of SimpleNN(
  (model): Sequential(
    (0): Linear(in_features=30, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
    (5): Sigmoid()
  )
)>

In [27]:
print(f'Final Loss: {loss.item():.4f}')

Final Loss: 0.1206
